# 04 – Natural Language Processing (NLP) Basics

Topics covered:
1. Text preprocessing: tokenisation, stemming, lemmatisation, stop-words
2. Bag of Words (BoW) & TF-IDF
3. Sentiment Analysis with Logistic Regression
4. Word Embeddings with Word2Vec / GloVe
5. Text Classification with LSTM (Keras)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re, string

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

np.random.seed(42)
tf.random.set_seed(42)

## 1. Text Preprocessing

In [ ]:
# Basic text cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)           # remove HTML tags
    text = re.sub(r'http\S+|www\S+', ' ', text)    # remove URLs
    text = re.sub(r'[^a-z\s]', ' ', text)          # keep only letters
    text = re.sub(r'\s+', ' ', text).strip()        # normalise whitespace
    return text

sample = "Hello! Check out https://example.com – it's <b>awesome</b> :)"
print('Original:', sample)
print('Cleaned :', clean_text(sample))

In [ ]:
# NLTK utilities (stopwords, stemming, lemmatisation)
try:
    import nltk
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt',     quiet=True)
    nltk.download('wordnet',   quiet=True)

    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer, WordNetLemmatizer
    from nltk.tokenize import word_tokenize

    text = 'The quick brown foxes are jumping over the lazy dogs'
    tokens = word_tokenize(text.lower())
    stop_words = set(stopwords.words('english'))
    tokens_filtered = [t for t in tokens if t not in stop_words and t.isalpha()]

    stemmer     = PorterStemmer()
    lemmatizer  = WordNetLemmatizer()
    stemmed     = [stemmer.stem(t) for t in tokens_filtered]
    lemmatized  = [lemmatizer.lemmatize(t) for t in tokens_filtered]

    print('Tokens   :', tokens_filtered)
    print('Stemmed  :', stemmed)
    print('Lemmatized:', lemmatized)

except ImportError:
    print('NLTK not installed: pip install nltk')

## 2. Bag of Words & TF-IDF

In [ ]:
corpus = [
    'I love machine learning and deep learning',
    'Deep learning is a subset of machine learning',
    'Natural language processing is fascinating',
    'I enjoy studying neural networks'
]

# Bag of Words
bow = CountVectorizer()
X_bow = bow.fit_transform(corpus)
print('BoW vocab size:', len(bow.vocabulary_))
pd.DataFrame(X_bow.toarray(), columns=bow.get_feature_names_out())

In [ ]:
# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
pd.DataFrame(X_tfidf.toarray().round(3), columns=tfidf.get_feature_names_out())

## 3. Text Classification with TF-IDF + Logistic Regression

In [ ]:
# 20 Newsgroups – 4 categories
cats = ['sci.space', 'rec.sport.hockey', 'talk.politics.guns', 'comp.graphics']
news = fetch_20newsgroups(subset='all', categories=cats, remove=('headers','footers','quotes'))

X_text = news.data
y_text = news.target

X_tr, X_te, y_tr, y_te = train_test_split(X_text, y_text, test_size=0.2, random_state=42, stratify=y_text)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=5))
])
pipe.fit(X_tr, y_tr)

print('Test Accuracy:', accuracy_score(y_te, pipe.predict(X_te)):.3f)
print(classification_report(y_te, pipe.predict(X_te), target_names=news.target_names))

In [ ]:
# Top TF-IDF features per class
feature_names = pipe['tfidf'].get_feature_names_out()
coef          = pipe['clf'].coef_

fig, axes = plt.subplots(1, len(cats), figsize=(20, 4))
for i, ax in enumerate(axes):
    top_idx = coef[i].argsort()[-15:][::-1]
    ax.barh(feature_names[top_idx][::-1], coef[i][top_idx][::-1], color='steelblue')
    ax.set_title(cats[i].split('.')[1])
plt.suptitle('Top Features per Class'); plt.tight_layout(); plt.show()

## 4. LSTM Text Classifier (IMDB Sentiment)

In [ ]:
VOCAB_SIZE  = 10000
MAX_LEN     = 256
EMBED_DIM   = 64

(X_train, y_train), (X_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

X_train = keras.preprocessing.sequence.pad_sequences(X_train, maxlen=MAX_LEN)
X_test  = keras.preprocessing.sequence.pad_sequences(X_test,  maxlen=MAX_LEN)

print('Train:', X_train.shape, '  Test:', X_test.shape)

In [ ]:
lstm_model = keras.Sequential([
    layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True),
    layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
    layers.Bidirectional(layers.LSTM(32)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
lstm_model.summary()

In [ ]:
history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=128,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

test_loss, test_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f'IMDB Test Accuracy: {test_acc:.3f}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, m in zip(axes, ['loss', 'accuracy']):
    ax.plot(history_lstm.history[m],         label='Train')
    ax.plot(history_lstm.history[f'val_{m}'], label='Val')
    ax.set_title(m.capitalize()); ax.legend()
plt.tight_layout(); plt.show()

## 5. Key Takeaways

| Technique | Use case |
|-----------|----------|
| BoW | Simple baseline; ignores word order |
| TF-IDF | Down-weights common words; better than BoW |
| n-grams | Capture short phrases |
| LSTM | Sequential models; captures long-range dependencies |
| Bidirectional LSTM | Context from both directions |
| Transformer / BERT | State-of-the-art NLP (see advanced resources) |

---

Congratulations – you have completed the **Beginner → Advanced** Machine Learning curriculum! 🎉